# D.R.O.N.A. - Notebook 09: LoRA fine-tune of Phi-3.5-mini (QLoRA)

Fine-tunes **Phi-3.5-mini-instruct** with **QLoRA (4-bit)** on the bias-aware advising Q&A dataset - Research Contribution **C2/C3**. Needs a **GPU** (Colab/Kaggle T4 is enough). ~30–60 min. Output: a LoRA adapter in `models/phi35-lora-advising/`.

**Using REAL curriculum/jobs?** Replace `data/raw/curriculum/*` and `data/manual_collection/*/*` in your repo before cell 3 (or commit them and re-clone).

## 1. Setup (edit `REPO_URL`)

In [ ]:
# ===========================================================================
#  D.R.O.N.A. - Colab / Kaggle setup.  *** RUN THIS CELL FIRST ***
#  Works on Google Colab and Kaggle. See docs/COLAB_TRAINING_GUIDE.md.
# ===========================================================================
import os, sys, subprocess, pathlib

# 1) GPU check ---------------------------------------------------------------
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "!!! NO GPU SELECTED.\n"
             "    Colab : Runtime > Change runtime type > T4 GPU\n"
             "    Kaggle: right panel > Settings > Accelerator > GPU T4 x2")

# 2) Get the D.R.O.N.A. repo -------------------------------------------------
#    EDIT REPO_URL to your GitHub repo. Private repo? use a read token:
#      https://<TOKEN>@github.com/<user>/D.R.O.N.A.git
#    OR leave it as-is and instead UPLOAD a zip of the project (Colab: Files panel)
#    / attach it as a Kaggle dataset named 'drona' - the search loop finds it.
REPO_URL = "https://github.com/<your-username>/D.R.O.N.A.git"   # <-- EDIT ME

def _is_repo(p):
    return pathlib.Path(p, "drona", "__init__.py").is_file()

search = ["D.R.O.N.A", ".", "/content/D.R.O.N.A",
          "/kaggle/working/D.R.O.N.A", "/kaggle/input/drona/D.R.O.N.A", "/kaggle/input/drona"]
repo = next((p for p in search if _is_repo(p)), None)
if repo is None and "<your-username>" not in REPO_URL:
    dest = "/content/D.R.O.N.A" if pathlib.Path("/content").is_dir() else "D.R.O.N.A"
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, dest], check=True)
    repo = dest
assert repo and _is_repo(repo), (
    "Repo not found. Set REPO_URL to your GitHub URL, OR upload/attach the project, "
    "then re-run. See docs/COLAB_TRAINING_GUIDE.md.")
os.chdir(repo)
print("repo:", os.getcwd())

# 3) Install the drona package (light deps; training deps are installed below)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=False)
print("setup complete -- continue to the next cell")

## 2. Install training dependencies

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.44", "peft>=0.12", "trl>=0.9.6",
                "accelerate>=0.33", "datasets>=2.18", "bitsandbytes>=0.43"], check=True)
print("training deps installed")

## 3. Build the SFT dataset
Runs the full data-prep chain (placeholder curriculum/jobs + real O*NET + grounded, bias-balanced Q&A). Replace the placeholders with real data first for a real model.

In [ ]:
subprocess.run([sys.executable, "scripts/prepare_training_data.py"], check=True)

import json
train = [json.loads(l) for l in open("data/finetune/sft_train.jsonl", encoding="utf-8")]
val   = [json.loads(l) for l in open("data/finetune/sft_val.jsonl", encoding="utf-8")]
print(f"SFT train={len(train)} val={len(val)}")
print(train[0]["text"][:400])

## 4. To a 🤗 `datasets.Dataset` (chat format)

In [ ]:
from datasets import Dataset
train_ds = Dataset.from_list([{"messages": e["messages"]} for e in train])
val_ds   = Dataset.from_list([{"messages": e["messages"]} for e in val])
train_ds

## 5. Load Phi-3.5-mini in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from drona.finetune.lora_config import DronaLoraConfig

cfg = DronaLoraConfig()
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    cfg.base_model, quantization_config=cfg.to_bnb_config(),
    device_map="auto", trust_remote_code=True)
model.config.use_cache = False
print(round(model.get_memory_footprint()/1e9, 2), "GB on GPU")

## 6. Train the LoRA adapter

In [ ]:
from peft import prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

model = prepare_model_for_kbit_training(model)
sft_args = SFTConfig(
    output_dir=cfg.output_dir,
    num_train_epochs=cfg.num_train_epochs,
    per_device_train_batch_size=cfg.per_device_train_batch_size,
    gradient_accumulation_steps=cfg.gradient_accumulation_steps,
    learning_rate=cfg.learning_rate,
    warmup_ratio=cfg.warmup_ratio,
    logging_steps=cfg.logging_steps,
    save_strategy=cfg.save_strategy,
    max_seq_length=cfg.max_seq_length,
    bf16=True, seed=cfg.seed, report_to=[])
trainer = SFTTrainer(
    model=model, args=sft_args,
    train_dataset=train_ds, eval_dataset=val_ds,
    peft_config=cfg.to_peft_config(), processing_class=tokenizer)
trainer.train()

## 7. Save the adapter

In [ ]:
trainer.save_model(cfg.output_dir)
tokenizer.save_pretrained(cfg.output_dir)
print("adapter saved to", cfg.output_dir)

## 8. (Optional) Export GGUF for Ollama
Run this only if you want a single-file model to serve locally with Ollama. It merges the adapter into the base weights and converts to GGUF via llama.cpp.

In [ ]:
# Merge adapter into base (needs ~8 GB GPU/CPU RAM):
# from peft import AutoPeftModelForCausalLM
# merged = AutoPeftModelForCausalLM.from_pretrained(cfg.output_dir, torch_dtype="auto")
# merged = merged.merge_and_unload(); merged.save_pretrained(cfg.output_dir + "-merged")
# tokenizer.save_pretrained(cfg.output_dir + "-merged")
# !git clone https://github.com/ggerganov/llama.cpp
# !pip -q install -r llama.cpp/requirements.txt
# !python llama.cpp/convert_hf_to_gguf.py {cfg.output_dir}-merged \
#     --outfile models/phi35-advising.gguf --outtype q4_k_m
print("see comments above to export GGUF")

## 9. Quick sanity check (base+LoRA generation)

In [ ]:
from transformers import pipeline
gen = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=400)
msg = train[0]["messages"][:2]
prompt = tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
print(gen(prompt)[0]["generated_text"][-800:])

## 10. Download the trained adapter

In [ ]:
# === Download your trained model back to your computer ===
# Edit SRC if you trained into a different directory.
import shutil, pathlib
SRC = "models/phi35-lora-advising"
base = "/content" if pathlib.Path("/content").is_dir() else "/kaggle/working"
zip_path = shutil.make_archive(f"{base}/drona_phi35_lora_adapter", "zip", SRC)
print("zipped:", zip_path, "->", round(pathlib.Path(zip_path).stat().st_size/1e6, 1), "MB")
try:
    from google.colab import files
    files.download(zip_path)                      # Colab: browser download starts
except Exception:
    print("Kaggle: open the right-hand 'Output' tab, find the .zip, and click Download.")
print("\nNext (on your PC): unzip into the repo so the paths match:\n  unzip drona_phi35_lora_adapter.zip -d models/phi35-lora-advising")